In [ ]:
# ==============================================================================
# Cell 1: Environment Setup and Hardware Initialization
# ==============================================================================
import os
import re
import warnings
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

from scipy.spatial.distance import directed_hausdorff
from sklearn.manifold import TSNE
from transformers import AutoTokenizer, AutoModel
from transformers.utils import logging

# Suppress non-critical warnings for cleaner output
logging.set_verbosity_error()
warnings.filterwarnings("ignore")

# ------------------------------------------------------------------------------
# Hardware Acceleration Detection (Apple Silicon Optimized)
# ------------------------------------------------------------------------------
def initialize_compute_device() -> torch.device:
    """
    Detects and returns the optimal hardware accelerator available.
    Prioritizes Apple Metal Performance Shaders (MPS), then CUDA, then CPU.
    """
    if torch.backends.mps.is_available():
        device = torch.device("mps")
        print("Hardware Accelerator: Apple Metal Performance Shaders (MPS)")
    elif torch.cuda.is_available():
        device = torch.device("cuda")
        print("Hardware Accelerator: NVIDIA CUDA")
    else:
        device = torch.device("cpu")
        print("Hardware Accelerator: CPU")
    return device

# ------------------------------------------------------------------------------
# Global Configuration
# ------------------------------------------------------------------------------
CONFIG = {
    "results_dir": "./results",
    "image_size": 256,
    "batch_size": 16,
    "num_workers": 0, 
    "learning_rate": 1e-4,
    "num_epochs": 15,
    "num_classes": 3,
    "embedding_dim": 512,
    "device": initialize_compute_device(),
}

# Ensure output directory exists
os.makedirs(CONFIG["results_dir"], exist_ok=True)
print(f"System initialized. Results will be saved to: {CONFIG['results_dir']}")

Hardware Accelerator: Apple Metal Performance Shaders (MPS)
System initialized. Results will be saved to: ./results


In [3]:
# ==============================================================================
# Cell 2: Performance Evaluation Metrics
# ==============================================================================

def compute_dice(predictions: torch.Tensor, targets: torch.Tensor, num_classes: int = 3, smooth: float = 1e-5) -> torch.Tensor:
    """
    Computes the macro-averaged Dice Similarity Coefficient.
    Ignores the background class (0) to focus on structural accuracy.
    """
    preds_argmax = torch.argmax(predictions, dim=1)
    scores = []
    
    for cls in range(1, num_classes):
        p = (preds_argmax == cls).float()
        t = (targets == cls).float()
        
        if t.sum() > 0:
            intersection = (p * t).sum()
            union = p.sum() + t.sum()
            scores.append((2.0 * intersection + smooth) / (union + smooth))
            
    return torch.mean(torch.stack(scores)) if scores else torch.tensor(1.0, device=predictions.device)

def compute_iou(predictions: torch.Tensor, targets: torch.Tensor, num_classes: int = 3, smooth: float = 1e-5) -> torch.Tensor:
    """
    Computes the Intersection over Union (Jaccard Index).
    Provides a stricter evaluation of boundary overlap than Dice.
    """
    preds_argmax = torch.argmax(predictions, dim=1)
    scores = []
    
    for cls in range(1, num_classes):
        p = (preds_argmax == cls).float()
        t = (targets == cls).float()
        
        if t.sum() > 0:
            intersection = (p * t).sum()
            union = p.sum() + t.sum() - intersection
            scores.append((intersection + smooth) / (union + smooth))
            
    return torch.mean(torch.stack(scores)) if scores else torch.tensor(1.0, device=predictions.device)

def compute_hausdorff(predictions: torch.Tensor, targets: torch.Tensor) -> float:
    """
    Computes the 95th percentile Hausdorff Distance (approximated via directed_hausdorff).
    Evaluates the worst-case boundary errors.
    """
    preds_np = torch.argmax(predictions, dim=1).cpu().numpy()
    targets_np = targets.cpu().numpy()
    batch_hd = []
    
    for b in range(preds_np.shape[0]):
        pred_coords = np.argwhere(preds_np[b] > 0)
        target_coords = np.argwhere(targets_np[b] > 0)
        
        if len(pred_coords) > 0 and len(target_coords) > 0:
            hd1 = directed_hausdorff(pred_coords, target_coords)[0]
            hd2 = directed_hausdorff(target_coords, pred_coords)[0]
            batch_hd.append(max(hd1, hd2))
        else:
            # Penalty for missing predictions or false positives on empty ground truth
            batch_hd.append(100.0)
            
    return np.mean(batch_hd)

In [4]:
# ==============================================================================
# Cell 3: 2.5D Multi-Modal Dataloader Infrastructure
# ==============================================================================

class PatientImageMapper:
    """
    Maps MRI volumes to their corresponding segmentation masks and clinical text reports.
    """
    def __init__(self, flair_img_dir: str, flair_mask_dir: str, text_dir: str):
        self.flair_img_dir = Path(flair_img_dir)
        self.flair_mask_dir = Path(flair_mask_dir)
        self.text_dir = Path(text_dir)

        self.img_files = sorted(self.flair_img_dir.glob("image_*.npy"))
        self.mask_files = sorted(self.flair_mask_dir.glob("mask_*.npy"))
        self.patient_dirs = sorted([d for d in self.text_dir.glob("BraTS20_*") if d.is_dir()])
        
        self.mapping = self._build_mapping()

    def _extract_id(self, pattern: str, string: str) -> int:
        match = re.search(pattern, string, re.IGNORECASE)
        return int(match.group(1)) if match else None

    def _build_mapping(self) -> dict:
        mapping = {}
        patient_dict = {self._extract_id(r'Training_(\d+)', d.name): d for d in self.patient_dirs if self._extract_id(r'Training_(\d+)', d.name) is not None}

        for img_file, mask_file in zip(self.img_files, self.mask_files):
            img_id = self._extract_id(r'image_(\d+)', img_file.name)
            if img_id is None: continue
            
            patient_id = img_id + 1
            if patient_id not in patient_dict: continue
            
            patient_dir = patient_dict[patient_id]
            txt_files = list(patient_dir.glob("*.txt"))
            
            mapping[img_id] = {
                "img_file": img_file, 
                "mask_file": mask_file,
                "patient_id": patient_dir.name,
                "text_file": txt_files[0] if txt_files else None
            }
        return mapping

    def get_sample(self, idx: int) -> dict:
        if idx not in self.mapping: return None
        info = self.mapping[idx]
        
        # Memory mapping is critical for handling large volumes sequentially
        image = np.load(info["img_file"], mmap_mode='r').astype(np.float32)
        mask = np.load(info["mask_file"], mmap_mode='r')
        text = "No clinical report available."
        
        if info["text_file"]:
            try:
                with open(info["text_file"], "r", encoding="utf-8", errors="ignore") as f:
                    text = f.read().strip()
            except Exception: pass
            
        return {"image": image, "mask": mask, "text": text, "patient_id": info["patient_id"], "idx": idx}

class CustomBRATSDataset(Dataset):
    """
    Extracts a 2.5D context stack (z-1, z, z+1) centered on the slice with maximum tumor volume.
    """
    def __init__(self, split: str = "train", image_size: int = 256, normalize: bool = True):
        self.image_size = image_size
        self.normalize = normalize

        img_dir = f"data/FLAIR_BRATS2020_split/{split}/images"
        mask_dir = f"data/FLAIR_BRATS2020_split/{split}/masks"

        self.mapper = PatientImageMapper(img_dir, mask_dir, "data/TextBRats/TextBraTSData")
        self.valid_indices = self._filter_valid_samples()
        print(f"[{split.upper()}] Initialized Dataset: {len(self.valid_indices)} valid patient volumes.")

    def _filter_valid_samples(self) -> list:
        valid = []
        for idx in self.mapper.mapping.keys():
            sample = self.mapper.get_sample(idx)
            if sample is None: continue
            mask = sample["mask"]
            
            # Extract central slice to check for minimum tumor presence
            middle_idx = mask.shape[2] // 2
            middle_mask = np.argmax(mask[:, :, middle_idx, :], axis=-1) if mask.ndim == 4 else mask[:, :, middle_idx]
            if np.sum(middle_mask > 0) > 50:
                valid.append(idx)
        return valid

    def __len__(self) -> int:
        return len(self.valid_indices)

    def _normalize(self, image: np.ndarray) -> np.ndarray:
        img_min, img_max = image.min(), image.max()
        return (image - img_min) / (img_max - img_min) if img_max > img_min else np.zeros_like(image).astype(np.float32)

    def _resize(self, slice_2d: np.ndarray, is_mask: bool = False) -> np.ndarray:
        pil_img = Image.fromarray((slice_2d * (1 if is_mask else 255)).astype(np.uint8))
        resample = Image.NEAREST if is_mask else Image.BILINEAR
        resized = pil_img.resize((self.image_size, self.image_size), resample)
        return np.array(resized).astype(np.int64) if is_mask else np.array(resized).astype(np.float32) / 255.0

    def __getitem__(self, idx: int) -> dict:
        real_idx = self.valid_indices[idx]
        sample = self.mapper.get_sample(real_idx)

        image, mask = sample["image"], sample["mask"]
        mask_volume = np.argmax(mask, axis=-1) if mask.ndim == 4 else mask
        
        # Locate the slice with the maximum tumor area
        slice_sums = [np.sum(mask_volume[:, :, i] > 0) for i in range(mask_volume.shape[2])]
        best_z = np.argmax(slice_sums)
        
        # Define 2.5D context bounds
        z_minus = max(0, best_z - 1)
        z_plus = min(mask_volume.shape[2] - 1, best_z + 1)

        img_z_minus = image[:, :, z_minus]
        img_z = image[:, :, best_z]
        img_z_plus = image[:, :, z_plus]
        mask_z = (mask_volume[:, :, best_z] > 0).astype(np.int64)

        if self.normalize:
            img_z_minus = self._normalize(img_z_minus)
            img_z = self._normalize(img_z)
            img_z_plus = self._normalize(img_z_plus)

        # Resize and stack into 3-channel input
        img_z_minus = self._resize(img_z_minus)
        img_z = self._resize(img_z)
        img_z_plus = self._resize(img_z_plus)
        mask_z = self._resize(mask_z, is_mask=True)

        image_25d = np.stack([img_z_minus, img_z, img_z_plus], axis=0)

        return {
            "image": torch.from_numpy(image_25d).float(),
            "mask": torch.from_numpy(mask_z).long(),
            "text": sample["text"],
            "patient_id": sample["patient_id"],
            "idx": real_idx
        }

In [5]:
# ==============================================================================
# Cell 4: Model Architecture Definitions
# ==============================================================================

class CLIPStyleTextEncoder(nn.Module):
    """
    Extracts semantic embeddings from clinical text reports using a pre-trained language model.
    """
    def __init__(self, model_name: str = 'sentence-transformers/all-MiniLM-L6-v2', embedding_dim: int = 512, max_length: int = 128):
        super().__init__()
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.text_model = AutoModel.from_pretrained(model_name)
        
        # Freeze transformer layers for computational efficiency
        for param in self.text_model.parameters(): 
            param.requires_grad = False
            
        self.projection = nn.Sequential(
            nn.Linear(self.text_model.config.hidden_size, 768), 
            nn.GELU(), 
            nn.Dropout(0.1), 
            nn.Linear(768, embedding_dim)
        )
        self.max_length = max_length

    def forward(self, texts: list) -> torch.Tensor:
        device = next(self.projection.parameters()).device
        encoded = {k: v.to(device) for k, v in self.tokenizer(list(texts), padding=True, truncation=True, max_length=self.max_length, return_tensors='pt').items()}
        
        outputs = self.text_model(**encoded)
        mask = encoded['attention_mask'].unsqueeze(-1).expand(outputs.last_hidden_state.size()).float()
        
        # Mean pooling over active tokens
        pooled = torch.sum(outputs.last_hidden_state * mask, dim=1) / torch.clamp(mask.sum(dim=1), min=1e-9)
        return F.normalize(self.projection(pooled), p=2, dim=-1)


class SegmentationBackbone(nn.Module):
    """
    Modified U-Net architecture. Accepts 3-channel input (2.5D context) and 
    exposes bottleneck features for latent alignment.
    """
    def __init__(self, in_channels: int = 3, out_channels: int = 3):
        super().__init__()
        
        # Encoder Pathway
        self.enc1 = nn.Sequential(nn.Conv2d(in_channels, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True), nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True))
        self.pool1 = nn.MaxPool2d(2)
        
        self.enc2 = nn.Sequential(nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True), nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True))
        self.pool2 = nn.MaxPool2d(2)
        
        self.enc3 = nn.Sequential(nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True), nn.Conv2d(256, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True))
        self.pool3 = nn.MaxPool2d(2)
        
        # Bottleneck (Feature extractor for multimodal alignment)
        self.bottleneck = nn.Sequential(nn.Conv2d(256, 512, 3, padding=1), nn.BatchNorm2d(512), nn.ReLU(inplace=True), nn.Conv2d(512, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True))
        
        # Decoder Pathway
        self.upconv3 = nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1)
        self.dec3 = nn.Sequential(nn.Conv2d(384, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True), nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True))
        
        self.upconv2 = nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1)
        self.dec2 = nn.Sequential(nn.Conv2d(192, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True))
        
        self.upconv1 = nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1)
        self.dec1 = nn.Sequential(nn.Conv2d(96, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True), nn.Conv2d(32, out_channels, 1))

    def forward(self, x: torch.Tensor) -> tuple:
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        
        b = self.bottleneck(self.pool3(e3))
        
        d3 = self.dec3(torch.cat([self.upconv3(b), e3], dim=1))
        d2 = self.dec2(torch.cat([self.upconv2(d3), e2], dim=1))
        logits = self.dec1(torch.cat([self.upconv1(d2), e1], dim=1))
        
        return logits, b


class MultimodalSegmentationModel(nn.Module):
    """
    Coordinates the image and text processing pipelines.
    """
    def __init__(self, num_classes: int = 3, embedding_dim: int = 512):
        super().__init__()
        self.seg_backbone = SegmentationBackbone(in_channels=3, out_channels=num_classes)
        self.text_encoder = CLIPStyleTextEncoder(embedding_dim=embedding_dim)
        
        # Maps spatial bottleneck features to the text embedding dimension
        self.img_proj = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)), 
            nn.Flatten(), 
            nn.Linear(256, 512), 
            nn.GELU(), 
            nn.Dropout(0.1), 
            nn.Linear(512, embedding_dim)
        )
    
    def forward(self, images: torch.Tensor, texts: list) -> dict:
        logits, spatial_features = self.seg_backbone(images)
        image_embeddings = F.normalize(self.img_proj(spatial_features), p=2, dim=-1)
        text_embeddings = self.text_encoder(texts)
        
        return {
            'segmentation': logits, 
            'image_embeddings': image_embeddings, 
            'text_embeddings': text_embeddings, 
            'features': spatial_features
        }


class MultiModalLoss(nn.Module):
    """
    Combines CrossEntropy for segmentation with symmetric CrossEntropy for contrastive text alignment.
    """
    def __init__(self, contrastive_weight: float = 0.1, temperature: float = 0.07):
        super().__init__()
        self.ce = nn.CrossEntropyLoss()
        self.weight = contrastive_weight
        self.temp = temperature

    def forward(self, out: dict, masks: torch.Tensor) -> dict:
        seg_loss = self.ce(out['segmentation'], masks.long())
        
        # Calculate cosine similarity matrix scaled by temperature
        logits = (out['image_embeddings'] @ out['text_embeddings'].T) / self.temp
        labels = torch.arange(logits.shape[0], device=logits.device)
        
        # Symmetric loss (Image-to-Text and Text-to-Image)
        contrastive_loss = (F.cross_entropy(logits, labels) + F.cross_entropy(logits.T, labels)) / 2
        
        total_loss = seg_loss + (self.weight * contrastive_loss)
        return {'total_loss': total_loss, 'seg_loss': seg_loss.item(), 'contrastive_loss': contrastive_loss.item()}

In [6]:
# ==============================================================================
# Cell 5: Visualization Routines
# ==============================================================================

def plot_training_curves(history: dict, save_dir: str):
    """Plots and saves the learning trajectories."""
    fig, ax = plt.subplots(1, 3, figsize=(18, 5))
    
    # Loss Plot
    ax[0].plot(history['train_loss'], label='Training Loss', color='blue')
    ax[0].plot(history['val_loss'], label='Validation Loss', color='red')
    ax[0].set_title('Objective Minimization', fontsize=12, fontweight='bold')
    ax[0].set_xlabel('Epoch'); ax[0].set_ylabel('Loss')
    ax[0].legend(); ax[0].grid(True, alpha=0.3)
    
    # Dice Plot
    ax[1].plot(history['train_dice'], label='Train Dice', color='green')
    ax[1].plot(history['val_dice'], label='Val Dice', color='purple')
    ax[1].set_title('Dice Similarity Coefficient', fontsize=12, fontweight='bold')
    ax[1].set_xlabel('Epoch'); ax[1].set_ylabel('Score')
    ax[1].legend(); ax[1].grid(True, alpha=0.3)
    
    # IoU Plot
    ax[2].plot(history['val_iou'], label='Validation IoU', color='orange')
    ax[2].set_title('Intersection over Union', fontsize=12, fontweight='bold')
    ax[2].set_xlabel('Epoch'); ax[2].set_ylabel('Score')
    ax[2].legend(); ax[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f"{save_dir}/01_training_metrics.png", dpi=300)
    plt.show()
 

def vis_seg(img_stack: np.ndarray, mask: np.ndarray, pred: np.ndarray, pid: str, save_dir: str, prefix: str = "02_seg"):
    """
    Visualizes the core slice, ground truth, and model prediction.
    img_stack: The (3, H, W) 2.5D input tensor.
    """
    # Extract the central Z-slice for visualization
    core_slice = img_stack[1]
    
    fig, ax = plt.subplots(1, 4, figsize=(18, 5))
    
    ax[0].imshow(core_slice, cmap='gray')
    ax[0].set_title('Input MRI (Core Slice)', fontweight='bold'); ax[0].axis('off')
    
    ax[1].imshow(mask, cmap='viridis')
    ax[1].set_title('Ground Truth Annotation', fontweight='bold'); ax[1].axis('off')
    
    ax[2].imshow(pred, cmap='viridis')
    ax[2].set_title('Model Prediction', fontweight='bold'); ax[2].axis('off')
    
    # Create red overlay mask
    overlay = np.stack([core_slice]*3, axis=-1)
    overlay[pred > 0] = [1.0, 0.0, 0.0] 
    
    ax[3].imshow(overlay)
    ax[3].set_title('Prediction Overlay', fontweight='bold'); ax[3].axis('off')
    
    plt.suptitle(f"Patient ID: {pid}", fontsize=14, y=1.05)
    plt.tight_layout()
    plt.savefig(f"{save_dir}/{prefix}_{pid}.png", bbox_inches='tight', dpi=300)
    plt.show()

In [7]:
# ==============================================================================
# Cell 6: Training Lifecycle Execution (macOS Compatible)
# ==============================================================================

def train_epoch(model: nn.Module, loader: DataLoader, opt: torch.optim.Optimizer, crit: nn.Module, dev: torch.device) -> tuple:
    model.train()
    total_loss, total_dice = 0.0, 0.0
    
    pbar = tqdm(loader, leave=False, desc="Training")
    for batch in pbar:
        img = batch["image"].to(dev)
        msk = batch["mask"].to(dev)
        txt = batch["text"]
        
        # Forward pass
        out = model(img, txt)
        loss_dict = crit(out, msk)
        
        # Backward pass
        opt.zero_grad()
        loss_dict["total_loss"].backward()
        opt.step()
        
        # Metric calculation
        batch_dice = compute_dice(out["segmentation"], msk).item()
        total_dice += batch_dice
        total_loss += loss_dict["total_loss"].item()
        
        pbar.set_postfix({"Loss": f"{loss_dict['total_loss'].item():.4f}", "Dice": f"{batch_dice:.4f}"})
        
    return total_loss / len(loader), total_dice / len(loader)


def validate(model: nn.Module, loader: DataLoader, crit: nn.Module, dev: torch.device, ret_scores: bool = False) -> tuple:
    model.eval()
    metrics = {"loss": 0.0, "dice": 0.0, "iou": 0.0, "hd": 0.0}
    individual_scores = []
    
    with torch.no_grad():
        for batch in tqdm(loader, leave=False, desc="Validating"):
            img = batch["image"].to(dev)
            msk = batch["mask"].to(dev)
            
            out = model(img, batch["text"])
            loss_dict = crit(out, msk)
            
            metrics["loss"] += loss_dict["total_loss"].item()
            metrics["dice"] += compute_dice(out['segmentation'], msk).item()
            metrics["iou"] += compute_iou(out['segmentation'], msk).item()
            metrics["hd"] += compute_hausdorff(out['segmentation'], msk)
            
            if ret_scores:
                for i in range(len(img)):
                    patient_id = batch["patient_id"][i]
                    pred_mask = torch.argmax(out['segmentation'][i:i+1], dim=1).squeeze(0).cpu()
                    dice_val = compute_dice(out['segmentation'][i:i+1], msk[i:i+1]).item()
                    
                    individual_scores.append({
                        "pid": patient_id, 
                        "dice": dice_val,
                        "img": img[i].cpu(), 
                        "mask": msk[i].cpu(), 
                        "pred": pred_mask
                    })
    
    averaged_metrics = {k: v / len(loader) for k, v in metrics.items()}
    return (averaged_metrics, individual_scores) if ret_scores else averaged_metrics


# ------------------------------------------------------------------------------
# Main Execution Block (Required for macOS Thread Safety)
# ------------------------------------------------------------------------------
if __name__ == '__main__':
    print("="*60)
    print("PHASE 1: DATA PIPELINE INITIALIZATION")
    print("="*60)
    
    # Initialize Loaders
    train_loader = DataLoader(CustomBRATSDataset("train", CONFIG["image_size"]), 
                              batch_size=CONFIG["batch_size"], shuffle=True, 
                              pin_memory=True, num_workers=CONFIG["num_workers"])
                              
    val_loader = DataLoader(CustomBRATSDataset("val", CONFIG["image_size"]), 
                            batch_size=CONFIG["batch_size"], shuffle=False, 
                            pin_memory=True, num_workers=CONFIG["num_workers"])

    # Initialize Model & Optimizer
    model = MultimodalSegmentationModel(CONFIG["num_classes"], CONFIG["embedding_dim"]).to(CONFIG["device"])
    optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG["learning_rate"])
    criterion = MultiModalLoss()

    # Trackers
    history = {k: [] for k in ["train_loss", "val_loss", "train_dice", "val_dice", "val_iou"]}
    best_val_dice = 0.0

    print("\n" + "="*60)
    print(f"PHASE 2: MODEL TRAINING (Target Epochs: {CONFIG['num_epochs']})")
    print("="*60)
    
    for epoch in range(CONFIG["num_epochs"]):
        print(f"\n[Epoch {epoch+1}/{CONFIG['num_epochs']}]")
        
        train_loss, train_dice = train_epoch(model, train_loader, optimizer, criterion, CONFIG["device"])
        val_metrics = validate(model, val_loader, criterion, CONFIG["device"])
        
        history["train_loss"].append(train_loss)
        history["train_dice"].append(train_dice)
        history["val_loss"].append(val_metrics["loss"])
        history["val_dice"].append(val_metrics["dice"])
        history["val_iou"].append(val_metrics["iou"])
        
        print(f"➜ Val Dice: {val_metrics['dice']:.4f} | Val IoU: {val_metrics['iou']:.4f} | Val HD95: {val_metrics['hd']:.2f}")
        
        # Model Checkpointing
        if val_metrics["dice"] > best_val_dice: 
            best_val_dice = val_metrics["dice"]
            save_path = f"{CONFIG['results_dir']}/best_model_weights.pt"
            torch.save(model.state_dict(), save_path)
            print(f"★ Saved new optimal weights to {save_path}")

    print("\n" + "="*60)
    print("PHASE 3: FINAL EVALUATION & VISUALIZATION")
    print("="*60)
    
    # Load optimal weights for final evaluation
    model.load_state_dict(torch.load(f"{CONFIG['results_dir']}/best_model_weights.pt"))
    final_metrics, score_details = validate(model, val_loader, criterion, CONFIG["device"], ret_scores=True)

    print("\n[Final Evaluation Metrics]")
    print(f"Average Dice Score: {final_metrics['dice']:.4f}")
    print(f"Average IoU Score:  {final_metrics['iou']:.4f}")
    print(f"Average HD95:       {final_metrics['hd']:.2f}")

    # Generate visual reports
    print("\nGenerating performance plots...")
    plot_training_curves(history, CONFIG["results_dir"])

    # Sort results to identify best and worst predictions
    score_details.sort(key=lambda x: x['dice'])

    print("\nRendering top 2 BEST predictions...")
    for i in range(-1, -3, -1): 
        patient = score_details[i]
        vis_seg(patient['img'].numpy(), patient['mask'].numpy(), patient['pred'].numpy(), 
                f"{patient['pid']} (Dice: {patient['dice']:.2f})", CONFIG["results_dir"], f"top_pred_{abs(i)}")

    print("\nRendering top 2 WORST predictions...")
    for i in range(2): 
        patient = score_details[i]
        vis_seg(patient['img'].numpy(), patient['mask'].numpy(), patient['pred'].numpy(), 
                f"{patient['pid']} (Dice: {patient['dice']:.2f})", CONFIG["results_dir"], f"bottom_pred_{i+1}")

    print("\n✅ Pipeline execution completed successfully. All artifacts saved to the results directory.")

PHASE 1: DATA PIPELINE INITIALIZATION
[TRAIN] Initialized Dataset: 230 valid patient volumes.
[VAL] Initialized Dataset: 80 valid patient volumes.


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10240.44it/s]



PHASE 2: MODEL TRAINING (Target Epochs: 15)

[Epoch 1/15]


Training:   0%|          | 0/15 [00:00<?, ?it/s]Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "<string>", line 1, in <module>
Traceback (most recent call last):
  File "/Users/praveentmr/.pyenv/versions/3.11.8/lib/python3.11/multiprocessing/spawn.py", line 122, in spawn_main
  File "<string>", line 1, in <module>
  File "<string>", line 1, in <module>
  File "/Users/praveentmr/.pyenv/versions/3.11.8/lib/python3.11/multiprocessing/spawn.py", line 122, in spawn_main
  File "/Users/praveentmr/.pyenv/versions/3.11.8/lib/python3.11/multiprocessing/spawn.py", line 122, in spawn_main
  File "/Users/praveentmr/.pyenv/versions/3.11.8/lib/python3.11/multiprocessing/spawn.py", line 122, in spawn_main
    exitcode = _main(fd, parent_sentinel)
    exitcode = _main(fd, parent_sentinel)
    exitcode = _main(fd, parent_sentinel)     exitcode = _main(fd, parent_sentinel) 
  
              ^ ^ ^ ^ ^ 

RuntimeError: DataLoader worker (pid(s) 68552, 68553) exited unexpectedly